# Data Cleaning & Preprocessing for Movie & TV Show Recommender

This notebook cleans a media dataset (such as IMDb/TMDb) and exports a clean, standardized dataset ready for the recommender web app.

- **Input file**: `data.csv`
- **Output file**: `data_clean.csv`

### Preprocessing Steps:
1. **Load Data**: Read `data.csv` into a Pandas DataFrame.
2. **Drop Votes**: Drop the `votes` column if present (not used for filtering).
3. **Handle Missing Values**: Drop rows missing critical fields (`title`, `year`, `genre`, `rating`).
4. **Split Year**: Extract `start_year` and `end_year` as integer values (handles both single-year movies like `2022` and series ranges like `2015-2022` or `2018- `).
5. **Clean Duration**: Extract integer duration in minutes.
6. **Clean Certificate**: Standardize certificate names, fill missing certificates with `'Unrated'`.
7. **Clean Genre & Stars**: Clean whitespace and parse actor names into clean comma-separated lists.
8. **Export Cleaned CSV**: Save the resulting DataFrame as `data_clean.csv`.

In [ ]:
import pandas as pd
import re
import ast

input_file = 'data.csv'
df = pd.read_csv(input_file)

print(f"Loaded '{input_file}' successfully with {len(df)} rows.")
df.head(3)

In [ ]:
if 'votes' in df.columns:
    df = df.drop(columns=['votes'])
    print("Dropped 'votes' column.")

print('Current columns:', df.columns.tolist())

Drop rows where essential values (`title`, `year`, `genre`, `rating`) are null because reccomendation depends totally on them. 

In [ ]:
essential_columns = ['title', 'year', 'genre', 'rating']
df = df.dropna(subset=[col for col in essential_columns if col in df.columns])
print(f'Rows remaining after removing missing essential values: {len(df)}')

### Extract `start_year` and `end_year`
- If single year (e.g., `(2022)`): `start_year = 2022`, `end_year = 2022`.
- If range (e.g., `(2015–2022)`): `start_year = 2015`, `end_year = 2022`.
- If ongoing (e.g., `(2018– )`): `start_year = 2018`, `end_year = 2024`.

In [ ]:
def parse_year(val):
    s = str(val)
    years = [int(y) for y in re.findall(r'\d{4}', s)]
    if len(years) >= 2:
        return years[0], years[1]
    elif len(years) == 1:
        # Check if it was an ongoing series (dash after year)
        if any(char in s for char in ['–', '-', ' ']) and s.find(str(years[0])) < len(s) - 5:
            return years[0], 2024
        return years[0], years[0]
    return None, None

year_parsed = df['year'].apply(parse_year)
df['start_year'] = [y[0] for y in year_parsed]
df['end_year'] = [y[1] for y in year_parsed]

# Remove rows where year could not be identified
df = df.dropna(subset=['start_year', 'end_year'])
df['start_year'] = df['start_year'].astype(int)
df['end_year'] = df['end_year'].astype(int)

print('Year extraction preview:')
df[['title', 'start_year', 'end_year']].head(3)

### Clean `certificate` and `duration`
- Fill missing certificate with `'Unrated'`.
- Extract integer duration in minutes (e.g., `'45 min'` -> `45`). Missing duration gets median.

In [ ]:
# Certificate
if 'certificate' in df.columns:
    df['certificate'] = df['certificate'].fillna('Unrated').astype(str).str.strip()
else:
    df['certificate'] = 'Unrated'

# Duration
def parse_duration(val):
    if pd.isna(val):
        return None
    match = re.search(r'\d+', str(val))
    return int(match.group()) if match else None

if 'duration' in df.columns:
    df['duration_min'] = df['duration'].apply(parse_duration)
    median_duration = int(df['duration_min'].median()) if not df['duration_min'].dropna().empty else 90
    df['duration_min'] = df['duration_min'].fillna(median_duration).astype(int)
else:
    df['duration_min'] = 90

print('Certificate and duration cleaned.')

### Clean `genre` and `stars`
- Clean genre tags (strip spaces).
- Clean actors / stars (remove director tags, brackets, and quotes).

In [ ]:
# Clean genres
if 'genre' in df.columns:
    df['genre'] = df['genre'].apply(lambda x: ', '.join([g.strip() for g in str(x).split(',') if g.strip()]))

# Clean stars
def clean_stars(val):
    if pd.isna(val):
        return 'Unknown'
    val_str = str(val)
    try:
        items = ast.literal_eval(val_str) if val_str.startswith('[') else val_str.split(',')
    except:
        items = val_str.split(',')
    
    cleaned = []
    has_stars_marker = any('Star' in str(it) for it in items)
    after_marker = False
    
    for it in items:
        s = str(it).replace('|', '').strip()
        if 'Star' in s:
            after_marker = True
            continue
        if has_stars_marker and not after_marker:
            continue
        s = s.strip("',\" \n\t")
        if s and s != '|':
            cleaned.append(s)
            
    return ', '.join(cleaned) if cleaned else 'Unknown'

if 'stars' in df.columns:
    df['stars'] = df['stars'].apply(clean_stars)
else:
    df['stars'] = 'Unknown'

df[['title', 'genre', 'stars']].head(3)

In [ ]:
# Drop original raw columns 'year' and 'duration'
for col in ['year', 'duration']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Reorder columns
preferred_order = ['title', 'start_year', 'end_year', 'certificate', 'duration_min', 'rating', 'genre', 'stars', 'description']
cols = [c for c in preferred_order if c in df.columns]
df = df[cols]

# Export
output_file = 'data_clean.csv'
df.to_csv(output_file, index=False)
print(f"Done! Successfully cleaned and saved {len(df)} records to '{output_file}'.")
df.head(5)